In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"))

In [ ]:
base_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"
print(os.listdir(base_path))

In [ ]:
dce_path = base_path + "/dce"

dce_files = os.listdir(dce_path)

print("Number of DCE files:", len(dce_files))
print("\nFirst 10 files:")

for file in dce_files[:10]:
    print(file)

In [ ]:
mask_path = base_path + "/mask"

mask_files = os.listdir(mask_path)

print("Number of mask files:", len(mask_files))

print("\nFirst 10 masks:")
for file in mask_files[:10]:
    print(file)

In [ ]:
from collections import Counter

patient_counts = Counter()

for file in dce_files:
    patient_id = file.split("_")[1]
    patient_counts[patient_id] += 1

print("Number of unique patients:", len(patient_counts))
print("\nDCE files per patient:")
print(Counter(patient_counts.values()))

In [ ]:
!pip install nibabel

In [ ]:
import nibabel as nib
import numpy as np

In [ ]:
patient_id = "1219"

patient_dce = [
    f for f in dce_files
    if f"ISPY1_{patient_id}_" in f
]

patient_mask = [
    f for f in mask_files
    if f"ISPY1_{patient_id}_" in f
]

print("DCE files:")
for f in patient_dce:
    print(f)

print("\nMask:")
for f in patient_mask:
    print(f)

In [ ]:
mri_file = patient_dce[0]
mri_path = dce_path + "/" + mri_file

mri = nib.load(mri_path)

print("File:", mri_file)
print("Shape:", mri.shape)
print("Data type:", mri.get_data_dtype())

In [ ]:
import matplotlib.pyplot as plt

middle = mri.shape[0] // 2

plt.imshow(mri.get_fdata()[middle], cmap="gray")
plt.title(f"Patient 1219 — slice {middle}")
plt.axis("off")
plt.show()

In [ ]:
mask_file = patient_mask[0]
mask_path_full = mask_path + "/" + mask_file

mask = nib.load(mask_path_full)

print("File:", mask_file)
print("Shape:", mask.shape)
print("Data type:", mask.get_data_dtype())

In [ ]:
mask_data = mask.get_fdata()

print("Unique values in mask:", np.unique(mask_data))

In [ ]:
print("MRI shape:  ", mri.shape)
print("Mask shape: ", mask.shape)

In [ ]:
mask_middle = mask_data[mask_data.shape[0] // 2]

plt.imshow(mask_middle, cmap="gray")
plt.title("Patient 1219 — Tumor Mask")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.imshow(mri.get_fdata()[14], cmap="gray")
plt.imshow(mask_data[14], cmap="Reds", alpha=0.4)

plt.title("Patient 1219 — MRI + Tumor Mask")
plt.axis("off")
plt.show()

In [ ]:
tumor_voxels_per_slice = np.sum(mask_data, axis=(1, 2))

for i, count in enumerate(tumor_voxels_per_slice):
    print(f"Slice {i:2d}: {int(count)} tumor voxels")

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(tumor_voxels_per_slice, marker="o")

plt.xlabel("Slice")
plt.ylabel("Tumor voxels")
plt.title("Tumor size across MRI slices — Patient 1219")
plt.grid(True)

plt.show()

In [ ]:
print("Voxel spacing:", mri.header.get_zooms())

In [ ]:
total_tumor_voxels = np.sum(tumor_voxels_per_slice)

voxel_spacing = mri.header.get_zooms()
voxel_volume = np.prod(voxel_spacing)

tumor_volume = total_tumor_voxels * voxel_volume

print("Total tumor voxels:", int(total_tumor_voxels))
print("Voxel volume:", voxel_volume, "mm³")
print("Tumor volume:", tumor_volume, "mm³")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, file in enumerate(patient_dce):
    path = dce_path + "/" + file
    volume = nib.load(path).get_fdata()

    axes[i].imshow(volume[14], cmap="gray")
    axes[i].set_title(f"DCE {i}")
    axes[i].axis("off")

plt.suptitle("Patient 1219 — Same slice across DCE acquisitions")
plt.show()

In [ ]:
for i, file in enumerate(patient_dce):
    volume = nib.load(dce_path + "/" + file).get_fdata()

    tumor_values = volume[mask_data == 1]

    print(f"DCE {i}:")
    print(f"  Mean intensity:   {tumor_values.mean():.2f}")
    print(f"  Median intensity: {np.median(tumor_values):.2f}")
    print()

# ==========================================
# Check MRI and mask shapes for all patients
# ==========================================

In [ ]:
from collections import defaultdict

shape_info = defaultdict(list)

for file in dce_files:
    path = dce_path + "/" + file
    img = nib.load(path)

    patient_id = file.split("_")[1]

    shape_info[patient_id].append(img.shape)

print("Number of patients checked:", len(shape_info))

# Show different shape patterns
unique_patterns = {}

for patient_id, shapes in shape_info.items():
    pattern = tuple(shapes)
    unique_patterns.setdefault(pattern, []).append(patient_id)

print("\nUnique DCE shape patterns:")
for pattern, patients in unique_patterns.items():
    print("Number of patients:", len(patients))
    print("Shapes:", pattern)

In [ ]:
# ==========================================
# Check DCE and mask shape consistency
# ==========================================

shape_mismatches = []

for patient_id, shapes in shape_info.items():

    # DCE shape
    dce_shape = shapes[0]

    # Find corresponding mask
    matching_mask = [
        f for f in mask_files
        if f"ISPY1_{patient_id}_" in f
    ]

    if len(matching_mask) != 1:
        shape_mismatches.append(
            (patient_id, "mask_missing_or_duplicate")
        )
        continue

    mask_img = nib.load(mask_path + "/" + matching_mask[0])
    mask_shape = mask_img.shape

    if dce_shape != mask_shape:
        shape_mismatches.append(
            (patient_id, dce_shape, mask_shape)
        )

print("Patients checked:", len(shape_info))
print("Shape mismatches:", len(shape_mismatches))

if shape_mismatches:
    print("\nMismatches:")
    for item in shape_mismatches:
        print(item)
else:
    print("All DCE volumes and masks have matching shapes.")

In [ ]:
dce_patient_ids = {
    f.split("_")[1]
    for f in dce_files
}

mask_patient_ids = {
    f.split("_")[1]
    for f in mask_files
}

print("Unique patients in DCE:", len(dce_patient_ids))
print("Unique patients in mask:", len(mask_patient_ids))

print("DCE only:", dce_patient_ids - mask_patient_ids)
print("Mask only:", mask_patient_ids - dce_patient_ids)

In [ ]:
# ==========================================
# Check voxel spacing across all patients
# ==========================================

spacing_info = {}

for patient_id, shapes in shape_info.items():

    # Get one DCE volume for this patient
    matching_dce = [
        f for f in dce_files
        if f"ISPY1_{patient_id}_" in f
    ]

    img = nib.load(dce_path + "/" + matching_dce[0])

    spacing_info[patient_id] = img.header.get_zooms()


# Find unique spacings
unique_spacings = set(spacing_info.values())

print("Number of unique voxel spacings:", len(unique_spacings))

for spacing in sorted(unique_spacings):
    count = sum(1 for s in spacing_info.values() if s == spacing)
    print(f"{spacing} → {count} patients")

In [ ]:
# ==========================================
# Check pCR availability in metadata
# ==========================================

print("Total metadata rows:", len(metadata))

print("\nUnique patient IDs in metadata:",
      metadata["pid"].nunique())

print("\npCR values:")
print(metadata["pCR"].value_counts(dropna=False))

In [ ]:
# ==========================================
# Match image patients with metadata
# ==========================================

# Patient IDs from the DCE dataset
dce_patient_ids_full = {
    f"ISPY1_{f.split('_')[1]}"
    for f in dce_files
}

# Patient IDs from metadata
metadata_patient_ids = set(metadata["pid"])

# Compare
image_only = dce_patient_ids_full - metadata_patient_ids
metadata_only = metadata_patient_ids - dce_patient_ids_full
common_patients = dce_patient_ids_full & metadata_patient_ids

print("Image patients:", len(dce_patient_ids_full))
print("Metadata patients:", len(metadata_patient_ids))
print("Common patients:", len(common_patients))
print("Image patients missing from metadata:", len(image_only))
print("Metadata patients without images:", len(metadata_only))

print("\nImage patients missing from metadata:")
print(sorted(image_only))

# Get metadata only for patients that actually have images
image_metadata = metadata[
    metadata["pid"].isin(dce_patient_ids_full)
]

print("\n pCR distribution for image patients:")
print(image_metadata["pCR"].value_counts(dropna=False))

In [ ]:
# ==========================================
# Check spatial alignment of DCE acquisitions
# ==========================================

affine_mismatches = []

for patient_id, shapes in shape_info.items():

    matching_dce = sorted([
        f for f in dce_files
        if f"ISPY1_{patient_id}_" in f
    ])

    # Load the three DCE volumes
    imgs = [
        nib.load(dce_path + "/" + f)
        for f in matching_dce
    ]

    # Compare affine matrices
    affine_0 = imgs[0].affine

    for i in range(1, len(imgs)):
        if not np.allclose(affine_0, imgs[i].affine):
            affine_mismatches.append(
                (patient_id, matching_dce[0], matching_dce[i])
            )
            break


print("Patients checked:", len(shape_info))
print("Patients with affine mismatches:", len(affine_mismatches))

if affine_mismatches:
    print("\nMismatches:")
    for item in affine_mismatches:
        print(item)
else:
    print("All three DCE acquisitions are spatially aligned.")

In [ ]:
# ==========================================
# Check DCE volumes for NaN / Inf values
# ==========================================

invalid_values = []

for patient_id, shapes in shape_info.items():

    matching_dce = sorted([
        f for f in dce_files
        if f"ISPY1_{patient_id}_" in f
    ])

    for filename in matching_dce:

        img = nib.load(dce_path + "/" + filename)
        data = img.get_fdata()

        nan_count = np.isnan(data).sum()
        inf_count = np.isinf(data).sum()

        if nan_count > 0 or inf_count > 0:
            invalid_values.append(
                (filename, nan_count, inf_count)
            )


print("DCE volumes checked:", len(dce_files))
print("Volumes containing NaN/Inf:", len(invalid_values))

if invalid_values:
    print("\nInvalid volumes:")
    for item in invalid_values:
        print(item)
else:
    print("No NaN or Inf values found in any DCE volume.")

In [ ]:
# ==========================================
# Explore DCE intensity ranges
# ==========================================

intensity_stats = []

for filename in dce_files:

    img = nib.load(dce_path + "/" + filename)
    data = img.get_fdata()

    intensity_stats.append({
        "filename": filename,
        "min": np.min(data),
        "max": np.max(data),
        "mean": np.mean(data),
        "median": np.median(data)
    })


intensity_df = pd.DataFrame(intensity_stats)

print("DCE volumes:", len(intensity_df))

print("\nOverall intensity statistics:")
print(intensity_df[["min", "max", "mean", "median"]].describe())

In [ ]:
# ==========================================
# Find volumes with very high intensity values
# ==========================================

high_intensity = intensity_df[
    intensity_df["max"] >= 60000
]

print("Volumes with max intensity >= 60000:",
      len(high_intensity))

print("\nThese volumes:")
print(high_intensity.to_string(index=False))

In [ ]:
# ==========================================
# Investigate the 65535 intensity value
# ==========================================

filename = "ISPY1_1187_spy1_vis1_dce_aqc_0.nii.gz"

img = nib.load(dce_path + "/" + filename)
data = img.get_fdata()

# Count exact 65535 values
count_65535 = np.sum(data == 65535)

total_voxels = data.size
percentage = (count_65535 / total_voxels) * 100

print("Shape:", data.shape)
print("Total voxels:", total_voxels)

print("\nVoxels with intensity 65535:", count_65535)
print("Percentage of volume:", percentage, "%")

# Look at high-intensity values
high_values = data[data > 1000]

print("\nVoxels with intensity > 1000:", len(high_values))

if len(high_values) > 0:
    print("Minimum value > 1000:", high_values.min())
    print("Maximum value > 1000:", high_values.max())

In [ ]:
# ==========================================
# Better visualization of extreme voxels
# ==========================================

middle_slice = data.shape[0] // 2
slice_data = data[middle_slice]

# Extreme voxels
extreme_mask = slice_data > 1000

# Use percentiles to display the normal MRI
vmin = np.percentile(slice_data, 1)
vmax = np.percentile(slice_data, 99)

print("Display range:")
print("1st percentile:", vmin)
print("99th percentile:", vmax)
print("Extreme voxels in middle slice:", extreme_mask.sum())

# MRI with sensible contrast
plt.figure(figsize=(7, 7))

plt.imshow(
    slice_data,
    cmap="gray",
    vmin=vmin,
    vmax=vmax
)

plt.title("Patient 1187 DCE 0 — Normal Intensity Window")
plt.axis("off")
plt.show()


# Extreme voxels alone
plt.figure(figsize=(7, 7))

plt.imshow(
    extreme_mask,
    cmap="gray"
)

plt.title("Patient 1187 DCE 0 — Voxels > 1000")
plt.axis("off")
plt.show()

In [ ]:
# ==========================================
# Find where the extreme voxels occur
# ==========================================

extreme_mask_3d = data > 1000

print("Extreme voxels per slice:\n")

for z in range(data.shape[0]):
    count = extreme_mask_3d[z].sum()

    if count > 0:
        print(f"Slice {z}: {count} voxels")

In [ ]:
# ==========================================
# Visualize extreme voxels on slice 1
# ==========================================

z = 1

slice_data = data[z]
extreme_mask = slice_data > 1000

print("Slice:", z)
print("Extreme voxels:", extreme_mask.sum())
print("Percentage of slice:",
      extreme_mask.mean() * 100, "%")

# Normal MRI view
vmin = np.percentile(slice_data, 1)
vmax = np.percentile(slice_data, 99)

plt.figure(figsize=(7, 7))

plt.imshow(
    slice_data,
    cmap="gray",
    vmin=vmin,
    vmax=vmax
)

plt.title("Patient 1187 DCE 0 — Slice 1")
plt.axis("off")
plt.show()


# Extreme-value locations
plt.figure(figsize=(7, 7))

plt.imshow(extreme_mask, cmap="gray")

plt.title("Patient 1187 DCE 0 — Slice 1, Voxels > 1000")
plt.axis("off")
plt.show()